<a href="https://colab.research.google.com/github/vitorbborges/ANTHEM/blob/develop/src/geolocate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Replace with your actual project path
%cd /content/drive/MyDrive/ANTHEM/src
!ls -a

In [ ]:
import geopandas as gpd
import zipfile
from pathlib import Path
from shapely.ops import substring, linemerge
from src.load_data import *
import tempfile
import shutil
import matplotlib.pyplot as plt

In [ ]:
data = load_var(subject_id=1, variable="CO2")
data['index'] = data.index
data

In [ ]:
kmz_path = 'data/raw_data/route.kmz'

# Create a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)

    # Extract the KMZ to the temp folder
    with zipfile.ZipFile(kmz_path, 'r') as z:
        z.extractall(tmpdir_path)

    # Read the KML file (usually named 'doc.kml')
    kml_path = tmpdir_path / 'doc.kml'
    gdf = gpd.read_file(kml_path, driver='KML')

In [ ]:
gdf

In [ ]:
route = gdf['geometry'][0]
points = gdf['geometry'][1:]

In [ ]:
# 1) grab your closed LineString
line = gdf.geometry.iloc[0]

# 2) extract & dedupe your 8 “cut” points, then project & sort them
pts = gdf.geometry.iloc[1:].drop_duplicates().to_list()
dists = sorted(line.project(pt) for pt in pts)

# 3) build the wrap‑around piece (last → first)
wrap = linemerge([
    substring(line, dists[-1], line.length),
    substring(line, 0.0,      dists[0])
])

# 4) build the in‑between pieces (1→2, 2→3, …)
segs = [wrap] + [
    substring(line, a, b)
    for a, b in zip(dists[:-1], dists[1:])
]

# 5) turn into a GeoSeries
segments_series = gpd.GeoSeries(segs, crs=gdf.crs)


In [ ]:
# create another column in a geodataframe segs
segments_df = gpd.GeoDataFrame({'geometry': segments_series})
segments_df['location'] = ["FG", "GH", "AB", "BC", "CD", "DE", "EF"]
segments_df

In [ ]:
S_dynamic = data[data["regime"] == "dynamic"]
S_dynamic

In [ ]:
import numpy as np
import geopandas as gpd

# 1) Merge S_dynamic with your segments GeoDataFrame
#    (assumes segments_df has columns ['location','geometry'])
sd = S_dynamic.merge(
    segments_df[['location','geometry']],
    on='location',
    how='left'
)

# 2) Prepare an empty column for the interpolated Points
sd['sample_pt'] = None

# 3) For each segment, evenly space the points
for loc, group in sd.groupby('location'):
    seg = group.geometry.iloc[0]         # the LineString for this location
    L   = seg.length                     # its total length
    N   = len(group)                     # how many samples to place

    # choose your scheme:
    #  - including endpoints: np.linspace(0   , L, N)
    #  - *excluding* endpoints:     np.linspace(0.5, L-0.5, N) /or/ (np.arange(N)+0.5)/N * L
    dists = np.linspace(0, L, N)         # here we *include* start & end

    # interpolate each distance into a Point
    pts = [seg.interpolate(d) for d in dists]

    # write them back into sd, preserving original row‐order
    sd.loc[group.index, 'sample_pt'] = pts

# 4) Extract X/Y (and Z, if present)
sd['x'] = sd['sample_pt'].apply(lambda p: p.x)
sd['y'] = sd['sample_pt'].apply(lambda p: p.y)
# if 3D:
# sd['z'] = sd['sample_pt'].apply(lambda p: p.z)

# Inspect
print(sd[['location','sample_pt','x','y']].head())


In [ ]:
sd

In [ ]:
fig, ax = plt.subplots()
for loc, group in sd.groupby('location'):
    ax.scatter(group['x'], group['y'], label=loc)
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
ax.set_title('Interpolated Points by Segment Location')
ax.legend(title='Location', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
kmz_path_pts = 'data/raw_data/points_of_interest.kmz'

# Create a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)

    # Extract the KMZ to the temp folder
    with zipfile.ZipFile(kmz_path_pts, 'r') as z:
        z.extractall(tmpdir_path)

    # Read the KML file (usually named 'doc.kml')
    kml_path = tmpdir_path / 'doc.kml'
    gdf_points = gpd.read_file(kml_path, driver='KML')

gdf_points.rename(columns={'Name': 'location'}, inplace=True)
gdf_points["x"] = gdf_points.geometry.x
gdf_points["y"] = gdf_points.geometry.y
gdf_points = gdf_points[['location', 'x', 'y']]

In [ ]:
S_static = data[data["regime"] == "static"]
S_static = S_static.merge(gdf_points, on="location", how="left").drop_duplicates()
S_static

In [ ]:
S_dynamic = sd.drop(columns=["sample_pt", "geometry"])
S_dynamic

In [ ]:
S = pd.concat([S_static, S_dynamic], ignore_index=True)
S.sort_values(by=["index"], inplace=True)
S.drop(columns=["index"], inplace=True)
S.to_csv("data/processed_data/S1CO2-approx-coordinates.csv", index=False)
S